In [1]:
# %%
import os
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import timm


/Users/nadinegunwan/Desktop/STAT440/STAT440-P2/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#paths of the file
PROJECT_DIR = Path(".").resolve()
TRAIN_DIR   = PROJECT_DIR / "train"
TEST_DIR    = PROJECT_DIR / "test"
LABELS_CSV  = TRAIN_DIR / "data.csv" 

print("PROJECT_DIR:", PROJECT_DIR)
print("TRAIN_DIR  :", TRAIN_DIR)
print("TEST_DIR   :", TEST_DIR)
print("LABELS_CSV :", LABELS_CSV)


PROJECT_DIR: /Users/nadinegunwan/Desktop/STAT440/STAT440-P2/Project
TRAIN_DIR  : /Users/nadinegunwan/Desktop/STAT440/STAT440-P2/Project/train
TEST_DIR   : /Users/nadinegunwan/Desktop/STAT440/STAT440-P2/Project/test
LABELS_CSV : /Users/nadinegunwan/Desktop/STAT440/STAT440-P2/Project/train/data.csv


In [ ]:
#read the csv and print it to see
df_all = pd.read_csv(LABELS_CSV)
print(df_all.head())
print("num rows:", len(df_all))
print("label counts:\n", df_all["label"].value_counts())


   index  label
0  K0000      0
1  K0001      0
2  K0002      1
3  K0003      0
4  K0004      1
Num rows: 1617
Label counts:
 label
1    1041
0     576
Name: count, dtype: int64


In [ ]:
#setting the class for train and test
class KilnTrainDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["index"]
        label = row["label"]
        
        img_path = self.img_dir / f"{img_id}.png"
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        label = torch.tensor(label, dtype=torch.float32)
        return image, label


class KilnTestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.ids = sorted([
            f.replace(".png", "")
            for f in os.listdir(self.img_dir)
            if f.endswith(".png")
        ])
    
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_path = self.img_dir / f"{img_id}.png"
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, img_id


In [ ]:
IMG_SIZE = 288
BATCH_SIZE = 32

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(
        IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

#splitting train and validation
train_df, val_df = train_test_split(
    df_all,
    test_size=0.2,
    random_state=42,
    stratify=df_all["label"],
)

len(train_df), len(val_df)


(1293, 324)

In [ ]:
#loader
train_ds = KilnTrainDataset(train_df, TRAIN_DIR, transform=train_tfms)
val_ds   = KilnTrainDataset(val_df,   TRAIN_DIR, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

len(train_loader), len(val_loader)


(41, 11)

In [ ]:
#Model: EfficientNet-B3 (binary head)
def create_effnet_b3_binary(pretrained: bool = True):
    model = timm.create_model(
        "efficientnet_b3",
        pretrained=pretrained,
        num_classes=1  
    )
    return model

model = create_effnet_b3_binary(pretrained=True).to(device)
model


EfficientNet(
  (conv_stem): Conv2d(3, 40, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=40, bias=False)
        (bn1): BatchNormAct2d(
          40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(40, 10, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(10, 40, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(40, 24, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (b

In [ ]:
#loss function and optimizer function
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-5,
)

N_EPOCHS = 10

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=N_EPOCHS
)


In [ ]:
#training and evaluation function definition
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        logits = model(images)         
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_probs = []
    all_trues = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)
            
            logits = model(images)
            loss = criterion(logits, labels)
            
            running_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_probs.extend(probs.cpu().numpy().ravel().tolist())
            all_trues.extend(labels.cpu().numpy().ravel().tolist())
    
    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    try:
        epoch_auc = roc_auc_score(all_trues, all_probs)
    except ValueError:
        epoch_auc = float("nan")
    
    return epoch_loss, epoch_acc, epoch_auc


In [ ]:
#training it 
OUT_DIR = PROJECT_DIR / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = OUT_DIR / "efficientnet_b3_pretrained_best.pt"

best_val_auc = -1.0

for epoch in range(1, N_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc, val_auc = evaluate(
        model, val_loader, criterion, device
    )
    
    scheduler.step()
    
    is_best = val_auc > best_val_auc
    if is_best:
        best_val_auc = val_auc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    
    print(
        f"Epoch {epoch:02d} | "
        f"Train: loss={train_loss:.4f}, acc={train_acc:.3f} | "
        f"Val: loss={val_loss:.4f}, acc={val_acc:.3f}, AUC={val_auc:.3f} "
        f"{'<- best' if is_best else ''}"
    )


Epoch 01 | Train: loss=1.4610, acc=0.640 | Val: loss=0.8710, acc=0.738, AUC=0.823 <-- best
Epoch 02 | Train: loss=0.5946, acc=0.828 | Val: loss=0.5861, acc=0.821, AUC=0.907 <-- best
Epoch 03 | Train: loss=0.3049, acc=0.891 | Val: loss=0.4431, acc=0.880, AUC=0.941 <-- best
Epoch 04 | Train: loss=0.2355, acc=0.916 | Val: loss=0.3266, acc=0.895, AUC=0.957 <-- best
Epoch 05 | Train: loss=0.1775, acc=0.937 | Val: loss=0.3118, acc=0.910, AUC=0.967 <-- best
Epoch 06 | Train: loss=0.1155, acc=0.963 | Val: loss=0.2908, acc=0.917, AUC=0.967 <-- best
Epoch 07 | Train: loss=0.1339, acc=0.954 | Val: loss=0.3092, acc=0.907, AUC=0.968 <-- best
Epoch 08 | Train: loss=0.0989, acc=0.963 | Val: loss=0.2944, acc=0.901, AUC=0.974 <-- best
Epoch 09 | Train: loss=0.0562, acc=0.979 | Val: loss=0.2654, acc=0.932, AUC=0.979 <-- best
Epoch 10 | Train: loss=0.0384, acc=0.988 | Val: loss=0.1779, acc=0.935, AUC=0.988 <-- best
Epoch 11 | Train: loss=0.0517, acc=0.981 | Val: loss=0.2328, acc=0.944, AUC=0.985 
Epoch 1

KeyboardInterrupt: 

In [ ]:
#load the best epoch model
best_model = create_effnet_b3_binary(pretrained=False).to(device)
best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
best_model.eval()

test_ds = KilnTestDataset(TEST_DIR, transform=val_tfms)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

len(test_ds), len(test_loader)


(724, 23)

In [ ]:
#data augmentation and take the average for stability 
all_ids = []
all_scores = []

with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device)
        
        #original
        logits1 = best_model(images)
        probs1 = torch.sigmoid(logits1)
        
        #horizontal flip
        images_h = torch.flip(images, dims=[3])  # flip width
        logits2 = best_model(images_h)
        probs2 = torch.sigmoid(logits2)
        
        #vertical flip
        images_v = torch.flip(images, dims=[2])  # flip height
        logits3 = best_model(images_v)
        probs3 = torch.sigmoid(logits3)
        
        #averaging probability
        probs = (probs1 + probs2 + probs3) / 3.0
        
        all_ids.extend(ids)
        all_scores.extend(probs.cpu().numpy().ravel().tolist())

len(all_ids), len(all_scores)


(724, 724)

In [ ]:
submission = pd.DataFrame({
    "index": all_ids,
    "score": all_scores,
})

submission = submission.sort_values("index")
SUBMISSION_PATH = PROJECT_DIR / "submission_effnet_b3_tta.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

submission.head(), SUBMISSION_PATH


(   index     score
 0  K1617  0.994422
 1  K1618  0.999996
 2  K1619  0.987212
 3  K1620  0.999983
 4  K1621  0.994111,
 PosixPath('/Users/nadinegunwan/Desktop/STAT440/STAT440-P2/Project/submission_effnet_b3_tta.csv'))